# SuttaPlayer Piper1 VITS Training Notebook
This notebook implements the versionless, clean-slate **Source of Truth** architecture for running your T4 GPU training runs.

### Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Establish terminal tools

In [ ]:
!pip install colab-xterm
%reload_ext colabxterm
%xterm

### Step 3: Copy Sutta Training Manager
Copy your versionless manager script directly from Drive to local `/content/` workspace.

In [ ]:
!cp /content/drive/MyDrive/sutta-tts-model-training/sutta-training-manager.ts /content/sutta-training-manager.ts

### Step 4: Environment Initialization (Compiling & Symlinking)
This command will spin up micromamba, create Python 3.11 virtual environment, compile Cython MAS align extensions, and symlink your Drive Git files locally.

In [ ]:
!deno run --allow-all /content/sutta-training-manager.ts --init

### Step 5: Start Training in the Foreground
Runs the PyTorch Lightning fit loop in the foreground of this cell. Streams logs in real-time, bypasses all browser keyboard conflicts, and prevents idle timeout.

In [ ]:
!deno run --allow-all /content/sutta-training-manager.ts --train-fg

### Step 6: Start Background Keep-Alive & Checkpoints Sync

In [ ]:
!nohup deno run --allow-all /content/sutta-training-manager.ts --monitor > /content/monitor.log 2>&1 &

### Step 7: Write & Execute Google Sheets Convergence Sync (Background)
Paste your Python sheet sync script below to capture checkpoints and write convergence metrics in real-time.

In [ ]:
%%writefile /content/sync_sheets.py
# Authenticate and start Sheets Sync Loop
import os
import time
import pandas as pd
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

csv_path = "/content/drive/MyDrive/piper_training/uat_metrics.csv"
sheet_name = "SuttaPlayer_UAT_Convergence"

print("🔍 Initializing Google Sheets UAT Sync...")
try:
    try:
        sh = gc.open(sheet_name)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(sheet_name)
        print(f"✅ Created new Google Sheet: '{sheet_name}' in your Drive.")
        
    ws = sh.get_worksheet(0)
    
    last_logged_row = len(ws.col_values(1)) if ws.col_values(1) else 0
    print(f"⚡ Active Sync daemon running. Monitoring CSV changes... (Last row logged: {last_logged_row})")
    
    while True:
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                total_rows = len(df)
                
                if total_rows > (last_logged_row - 1 if last_logged_row > 0 else 0):
                    start_idx = max(0, last_logged_row - 1 if last_logged_row > 0 else 0)
                    new_data = df.iloc[start_idx:].values.tolist()
                    
                    if last_logged_row == 0:
                        ws.append_row(df.columns.tolist())
                        last_logged_row += 1
                        
                    for row in new_data:
                        ws.append_row(row)
                        print(f"  [Sheets Sync] Appended Epoch {row[1]} | Loss: {row[2]} to Sheet.")
                        
                    last_logged_row = total_rows + 1
            except Exception as e:
                print(f"  [WARNING] Read/write collision on CSV (training is writing). Retrying next tick. Error: {e}")
        time.sleep(15)
except KeyboardInterrupt:
    print("\n⏹️ Sheets Sync Stopped.")

In [ ]:
!nohup python3 -u /content/sync_sheets.py > /content/sheets_sync.log 2>&1 &

### Progress & Log Checking (Zero Lag)

In [ ]:
!tail -n 20 /content/sheets_sync.log

In [ ]:
!tail -n 20 /content/monitor.log

### Environment Backup (Saves 3 Mins Compile Time)

In [ ]:
!tar -czf /content/drive/MyDrive/sutta-tts-model-training/py311_env_backup.tar.gz -C /root/micromamba/envs py311

### Instant Environment Restore (Clean Session Restore under 30 seconds!)

In [ ]:
!mkdir -p /root/micromamba/envs
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
!tar -xzf /content/drive/MyDrive/sutta-tts-model-training/py311_env_backup.tar.gz -C /root/micromamba/envs
!deno run --allow-all /content/sutta-training-manager.ts --diag-setup